In [ ]:
import os
import time
import glob
import shutil
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
warnings.filterwarnings("ignore")
gpus=tf.config.list_physical_devices("GPU")
for g in gpus:
    tf.config.experimental.set_memory_growth(g,True)
from tensorflow import keras
from tensorflow.keras import layers
from skimage.metrics import structural_similarity
plt.rcParams["legend.framealpha"]=1.0
seed=42
keras.utils.set_random_seed(seed)
out="/kaggle/working/outputs"
os.makedirs(out,exist_ok=True)
print("TensorFlow",tf.__version__)
print("Keras",keras.__version__)
print("GPUs",gpus)
if len(gpus)==0:
    print("WARNING: no GPU detected. Set Accelerator to GPU T4 x2 or P100 in the Kaggle notebook settings.")

In [ ]:
epochs_fc=20
epochs_cae=20
epochs_dn=30
epochs_vae=30
epochs_study=20
batch=128
lr=1e-3
levels_g=[0.1,0.2,0.3]
levels_sp=[0.05,0.10,0.20]
latent_dims=[2,8,16,32]

In [ ]:
def load_mnist():
    try:
        return keras.datasets.mnist.load_data()
    except Exception:
        paths=glob.glob("/kaggle/input/**/mnist.npz",recursive=True)
        if len(paths)==0:
            raise RuntimeError("Turn on Internet in the Kaggle settings or attach an MNIST dataset that contains mnist.npz")
        d=np.load(paths[0])
        return (d["x_train"],d["y_train"]),(d["x_test"],d["y_test"])

(xa,ya),(xb,yb)=load_mnist()
x_train_full=(xa[:10000].astype("float32")/255.0)[...,None]
y_train_full=ya[:10000]
x_test=(xb[:2000].astype("float32")/255.0)[...,None]
y_test=yb[:2000]
x_tr=x_train_full[:9000]
x_val=x_train_full[9000:]
print("train",x_tr.shape,"val",x_val.shape,"test",x_test.shape)
print("pixel range",x_train_full.min(),x_train_full.max())

In [ ]:
def ssim_each(x,xh):
    return np.array([structural_similarity(x[i,:,:,0],np.clip(xh[i,:,:,0],0,1),data_range=1.0) for i in range(len(x))])

def score(x,xh):
    return {"MSE":float(np.mean((x-xh)**2)),"MAE":float(np.mean(np.abs(x-xh))),"SSIM":float(np.mean(ssim_each(x,xh)))}

def per_img_mse(x,xh):
    return np.mean((x-xh)**2,axis=(1,2,3))

def flat(x):
    return x.reshape(len(x),-1)

def unflat(x):
    return x.reshape(len(x),28,28,1)

def save_fig(name):
    plt.tight_layout()
    plt.savefig(os.path.join(out,name+".eps"),format="eps",bbox_inches="tight")
    plt.show()
    plt.close()

def show(df,title):
    print(title)
    print(df.to_string(index=False))
    print()

def fit_model(model,xi,xt,xvi,xvt,epochs):
    t0=time.time()
    h=model.fit(xi,xt,validation_data=(xvi,xvt),epochs=epochs,batch_size=batch,verbose=2)
    return h.history,time.time()-t0

def plot_rows(arrs,labels,name,n=10,cmaps=None):
    r=len(arrs)
    if cmaps is None:
        cmaps=["gray"]*r
    fig,ax=plt.subplots(r,n,figsize=(1.3*n,1.5*r+0.4),squeeze=False)
    for i in range(r):
        for j in range(n):
            ax[i,j].imshow(arrs[i][j,:,:,0],cmap=cmaps[i],vmin=0,vmax=1)
            ax[i,j].axis("off")
        ax[i,n//2].set_title(labels[i],fontsize=10)
    save_fig(name)

def plot_grid(imgs,rows,cols,name,title):
    fig,ax=plt.subplots(rows,cols,figsize=(1.6*cols,1.6*rows),squeeze=False)
    for i in range(rows):
        for j in range(cols):
            ax[i,j].imshow(imgs[i*cols+j,:,:,0],cmap="gray",vmin=0,vmax=1)
            ax[i,j].axis("off")
    fig.suptitle(title)
    save_fig(name)

def plot_latent_scatter(z,labels,name,title,xl="z1",yl="z2"):
    fig,ax=plt.subplots(figsize=(7,6))
    sc=ax.scatter(z[:,0],z[:,1],c=labels,cmap="tab10",s=10)
    cb=fig.colorbar(sc,ticks=range(10))
    cb.set_label("Digit label")
    ax.set_xlabel(xl)
    ax.set_ylabel(yl)
    ax.set_title(title)
    ax.grid(True)
    save_fig(name)

def loss_summary(hist,tk,vk,tag):
    tr=np.array(hist[tk])
    va=np.array(hist[vk])
    be=int(np.argmin(va))+1
    print(tag)
    print("Final train loss",round(float(tr[-1]),5))
    print("Final validation loss",round(float(va[-1]),5))
    print("Best validation loss",round(float(va.min()),5),"at epoch",be)
    print("Final gap (val - train)",round(float(va[-1]-tr[-1]),5))
    print("Train loss change over last 5 epochs",round(float(tr[-1]-tr[-6]),5))
    print("Validation loss change over last 5 epochs",round(float(va[-1]-va[-6]),5))
    print("Validation rising after best epoch",bool(va[-1]>va.min()*1.01))
    print()

def latent_stats(z,labels,tag):
    print(tag)
    print("Latent mean",np.round(z.mean(axis=0),4),"std",np.round(z.std(axis=0),4))
    print("Latent min",np.round(z.min(axis=0),4),"max",np.round(z.max(axis=0),4))
    cents=np.array([z[labels==d].mean(axis=0) for d in range(10)])
    stds=np.array([z[labels==d].std(axis=0) for d in range(10)])
    rows=[]
    for d in range(10):
        row={"Digit":d}
        for k in range(z.shape[1]):
            row["Centroid z"+str(k+1)]=round(float(cents[d,k]),3)
        for k in range(z.shape[1]):
            row["Std z"+str(k+1)]=round(float(stds[d,k]),3)
        rows.append(row)
    print(pd.DataFrame(rows).to_string(index=False))
    dist=np.linalg.norm(z[:,None,:]-cents[None,:,:],axis=2)
    pred=np.argmin(dist,axis=1)
    print("Nearest-centroid digit accuracy in latent space",round(float(np.mean(pred==labels)),4))
    cd=np.linalg.norm(cents[:,None,:]-cents[None,:,:],axis=2)
    off=cd[np.triu_indices(10,1)]
    print("Mean distance between digit centroids",round(float(off.mean()),4),"min",round(float(off.min()),4))
    print("Mean within-digit spread",round(float(np.mean([np.linalg.norm(z[labels==d]-cents[d],axis=1).mean() for d in range(10)])),4))
    print()

In [ ]:
def build_fc(dz=16):
    inp=keras.Input(shape=(784,))
    h=layers.Dense(128,activation="relu")(inp)
    h=layers.Dense(32,activation="relu")(h)
    z=layers.Dense(dz,name="latent")(h)
    h=layers.Dense(32,activation="relu")(z)
    h=layers.Dense(128,activation="relu")(h)
    o=layers.Dense(784,activation="sigmoid")(h)
    m=keras.Model(inp,o,name="fc_ae_"+str(dz))
    e=keras.Model(inp,z,name="fc_enc_"+str(dz))
    m.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),loss="binary_crossentropy",jit_compile=False)
    return m,e

fc,fc_enc=build_fc(16)
fc.summary()
hist_fc,time_fc=fit_model(fc,flat(x_tr),flat(x_tr),flat(x_val),flat(x_val),epochs_fc)
xh_fc=unflat(fc.predict(flat(x_test),batch_size=256,verbose=0))
res_fc=score(x_test,xh_fc)
params_fc=fc.count_params()
test_bce_fc=fc.evaluate(flat(x_test),flat(x_test),batch_size=256,verbose=0)
print("FC AE training time (s)",round(time_fc,2))
print("FC AE test BCE",round(float(test_bce_fc),5))
show(pd.DataFrame([{"Metric":"Test MSE","Value":round(res_fc["MSE"],6)},{"Metric":"Test MAE","Value":round(res_fc["MAE"],6)},{"Metric":"Mean SSIM","Value":round(res_fc["SSIM"],6)}]),"Basic autoencoder (FC, latent 16) test metrics")

In [ ]:
plot_rows([x_test,xh_fc,np.abs(x_test-xh_fc)],["Original","FC-AE Reconstruction","Absolute Error"],"plot01_original_vs_reconstructed_fcae",n=10,cmaps=["gray","gray","hot"])

ep=np.arange(1,len(hist_fc["loss"])+1)
plt.figure(figsize=(7,4.5))
plt.plot(ep,hist_fc["loss"],label="Training loss")
plt.plot(ep,hist_fc["val_loss"],label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Reconstruction loss (binary cross-entropy)")
plt.title("FC autoencoder training and validation loss")
plt.legend()
plt.grid(True)
save_fig("plot02_fcae_train_val_loss")
loss_summary(hist_fc,"loss","val_loss","FC autoencoder loss summary")
print("Per-image MSE on the 10 displayed test images",np.round(per_img_mse(x_test[:10],xh_fc[:10]),5))
print("Per-image SSIM on the 10 displayed test images",np.round(ssim_each(x_test[:10],xh_fc[:10]),4))

In [ ]:
def build_cae():
    inp=keras.Input(shape=(28,28,1))
    h=layers.Conv2D(32,3,activation="relu",padding="same")(inp)
    h=layers.MaxPooling2D(2,padding="same")(h)
    h=layers.Conv2D(64,3,activation="relu",padding="same")(h)
    h=layers.MaxPooling2D(2,padding="same")(h)
    z=layers.Conv2D(64,3,activation="relu",padding="same")(h)
    h=layers.UpSampling2D(2)(z)
    h=layers.Conv2D(32,3,activation="relu",padding="same")(h)
    h=layers.UpSampling2D(2)(h)
    o=layers.Conv2D(1,3,activation="sigmoid",padding="same")(h)
    m=keras.Model(inp,o)
    m.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),loss="binary_crossentropy",jit_compile=False)
    return m

cae=build_cae()
cae.summary()
hist_cae,time_cae=fit_model(cae,x_tr,x_tr,x_val,x_val,epochs_cae)
xh_cae=cae.predict(x_test,batch_size=256,verbose=0)
res_cae=score(x_test,xh_cae)
params_cae=cae.count_params()
print("CAE training time (s)",round(time_cae,2))
loss_summary(hist_cae,"loss","val_loss","Convolutional autoencoder loss summary")
cmp_df=pd.DataFrame([
{"Model":"Fully Connected AE","MSE":round(res_fc["MSE"],6),"MAE":round(res_fc["MAE"],6),"SSIM":round(res_fc["SSIM"],6),"Parameters":params_fc},
{"Model":"Convolutional AE","MSE":round(res_cae["MSE"],6),"MAE":round(res_cae["MAE"],6),"SSIM":round(res_cae["SSIM"],6),"Parameters":params_cae}])
show(cmp_df,"Fully connected versus convolutional autoencoder")
plot_rows([x_test,xh_fc,xh_cae],["Original","FC-AE Reconstruction","CAE Reconstruction"],"plot03_fcae_vs_cae_reconstruction",n=10)
plot_rows([x_test,xh_cae,np.abs(x_test-xh_cae)],["Original","CAE Reconstruction","Absolute Error"],"plot01b_original_vs_reconstructed_cae",n=10,cmaps=["gray","gray","hot"])

In [ ]:
def gauss_noise(x,s,r):
    s=np.asarray(s,dtype="float32")
    return np.clip(x+r.normal(0,1,x.shape).astype("float32")*s,0,1).astype("float32")

def sp_noise(x,p,r):
    p=np.asarray(p,dtype="float32")
    u=r.random(x.shape).astype("float32")
    y=x.copy()
    y[u<p/2]=0.0
    y[(u>=p/2)&(u<p)]=1.0
    return y

rn=np.random.default_rng(1)
xn_tr_g=gauss_noise(x_tr,rn.choice(levels_g,size=(len(x_tr),1,1,1)),rn)
xn_val_g=gauss_noise(x_val,rn.choice(levels_g,size=(len(x_val),1,1,1)),rn)
xn_tr_s=sp_noise(x_tr,rn.choice(levels_sp,size=(len(x_tr),1,1,1)),rn)
xn_val_s=sp_noise(x_val,rn.choice(levels_sp,size=(len(x_val),1,1,1)),rn)

dcae_g=build_cae()
hist_dg,time_dg=fit_model(dcae_g,xn_tr_g,x_tr,xn_val_g,x_val,epochs_dn)
print("Gaussian denoising CAE training time (s)",round(time_dg,2))
loss_summary(hist_dg,"loss","val_loss","Gaussian denoising CAE loss summary")

dcae_s=build_cae()
hist_ds,time_ds=fit_model(dcae_s,xn_tr_s,x_tr,xn_val_s,x_val,epochs_dn)
print("Salt-and-pepper denoising CAE training time (s)",round(time_ds,2))
loss_summary(hist_ds,"loss","val_loss","Salt-and-pepper denoising CAE loss summary")

def eval_denoiser(model,fn,levels,base):
    rows=[]
    outs={}
    for i,l in enumerate(levels):
        r=np.random.default_rng(base+i)
        xn=fn(x_test,l,r)
        xh=model.predict(xn,batch_size=256,verbose=0)
        a=score(x_test,xh)
        b=score(x_test,xn)
        rows.append({"Noise Level":l,"MSE":round(a["MSE"],6),"MAE":round(a["MAE"],6),"SSIM":round(a["SSIM"],6),"Noisy-input MSE":round(b["MSE"],6),"Noisy-input MAE":round(b["MAE"],6),"Noisy-input SSIM":round(b["SSIM"],6)})
        outs[l]=(xn,xh)
    return pd.DataFrame(rows),outs

df_g,outs_g=eval_denoiser(dcae_g,gauss_noise,levels_g,100)
df_s,outs_s=eval_denoiser(dcae_s,sp_noise,levels_sp,200)
show(df_g,"Gaussian noise: denoising CAE quality against clean test images (sigma)")
show(df_s,"Salt-and-pepper noise: denoising CAE quality against clean test images (p)")

for l in levels_g:
    plot_rows([x_test,outs_g[l][0],outs_g[l][1]],["Clean","Noisy (Gaussian sigma="+str(l)+")","Denoised"],"plot04_gaussian_"+str(l).replace(".","p"),n=10)
for l in levels_sp:
    plot_rows([x_test,outs_s[l][0],outs_s[l][1]],["Clean","Noisy (Salt-and-pepper p="+str(l)+")","Denoised"],"plot04_saltpepper_"+str(l).replace(".","p"),n=10)

def plot_noise_curves(df,xl,title,name):
    fig,ax=plt.subplots(1,3,figsize=(15,4.2))
    for a,m in zip(ax,["MSE","MAE","SSIM"]):
        a.plot(df["Noise Level"],df[m],marker="o",label="Denoised")
        a.plot(df["Noise Level"],df["Noisy-input "+m],marker="s",linestyle="--",label="Noisy input")
        a.set_xlabel(xl)
        a.set_ylabel(m)
        a.set_title(m+" versus noise level")
        a.grid(True)
        a.legend()
    fig.suptitle(title)
    save_fig(name)

plot_noise_curves(df_g,"Gaussian noise standard deviation","Denoising quality under Gaussian noise","plot05_noise_level_vs_quality_gaussian")
plot_noise_curves(df_s,"Salt-and-pepper corruption proportion","Denoising quality under salt-and-pepper noise","plot05_noise_level_vs_quality_saltpepper")

xn_ref,xh_dn=outs_g[0.2]
res_dn=score(x_test,xh_dn)
params_dn=dcae_g.count_params()
print("Gaussian denoising CAE: SSIM improvement over noisy input at each level",np.round(df_g["SSIM"].values-df_g["Noisy-input SSIM"].values,4))
print("Gaussian denoising CAE: MSE reduction factor at each level",np.round(df_g["Noisy-input MSE"].values/df_g["MSE"].values,3))
print("Salt-and-pepper denoising CAE: SSIM improvement over noisy input at each level",np.round(df_s["SSIM"].values-df_s["Noisy-input SSIM"].values,4))
print("Salt-and-pepper denoising CAE: MSE reduction factor at each level",np.round(df_s["Noisy-input MSE"].values/df_s["MSE"].values,3))

In [ ]:
class Sampling(layers.Layer):
    def call(self,inputs):
        zm,zl=inputs
        eps=tf.random.normal(shape=tf.shape(zm))
        return zm+tf.exp(0.5*zl)*eps
    def compute_output_shape(self,input_shape):
        return input_shape[0]

class VAE(keras.Model):
    def __init__(self,encoder,decoder,beta=1.0,**kw):
        super().__init__(**kw)
        self.encoder=encoder
        self.decoder=decoder
        self.beta=beta
        self.loss_tracker=keras.metrics.Mean(name="loss")
        self.rec_tracker=keras.metrics.Mean(name="rec")
        self.kl_tracker=keras.metrics.Mean(name="kl")
    @property
    def metrics(self):
        return [self.loss_tracker,self.rec_tracker,self.kl_tracker]
    def call(self,x):
        return self.decoder(self.encoder(x)[2])
    def losses_for(self,x,training):
        zm,zl,z=self.encoder(x,training=training)
        xr=self.decoder(z,training=training)
        rec=tf.reduce_mean(tf.reduce_sum(keras.losses.binary_crossentropy(x,xr),axis=(1,2)))
        kl=tf.reduce_mean(-0.5*tf.reduce_sum(1+zl-tf.square(zm)-tf.exp(zl),axis=1))
        return rec,kl
    def train_step(self,data):
        x=data[0] if isinstance(data,(tuple,list)) else data
        with tf.GradientTape() as tape:
            rec,kl=self.losses_for(x,True)
            loss=rec+self.beta*kl
        grads=tape.gradient(loss,self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads,self.trainable_variables))
        self.loss_tracker.update_state(loss)
        self.rec_tracker.update_state(rec)
        self.kl_tracker.update_state(kl)
        return {"loss":self.loss_tracker.result(),"rec":self.rec_tracker.result(),"kl":self.kl_tracker.result()}
    def test_step(self,data):
        x=data[0] if isinstance(data,(tuple,list)) else data
        rec,kl=self.losses_for(x,False)
        loss=rec+self.beta*kl
        self.loss_tracker.update_state(loss)
        self.rec_tracker.update_state(rec)
        self.kl_tracker.update_state(kl)
        return {"loss":self.loss_tracker.result(),"rec":self.rec_tracker.result(),"kl":self.kl_tracker.result()}

def build_vae(dz=2):
    ei=keras.Input(shape=(28,28,1))
    h=layers.Conv2D(32,3,strides=2,padding="same",activation="relu")(ei)
    h=layers.Conv2D(64,3,strides=2,padding="same",activation="relu")(h)
    h=layers.Flatten()(h)
    h=layers.Dense(16,activation="relu")(h)
    zm=layers.Dense(dz,name="z_mean")(h)
    zl=layers.Dense(dz,name="z_log_var")(h)
    z=Sampling()([zm,zl])
    enc=keras.Model(ei,[zm,zl,z],name="encoder")
    di=keras.Input(shape=(dz,))
    d=layers.Dense(7*7*64,activation="relu")(di)
    d=layers.Reshape((7,7,64))(d)
    d=layers.Conv2DTranspose(64,3,strides=2,padding="same",activation="relu")(d)
    d=layers.Conv2DTranspose(32,3,strides=2,padding="same",activation="relu")(d)
    do=layers.Conv2DTranspose(1,3,padding="same",activation="sigmoid")(d)
    dec=keras.Model(di,do,name="decoder")
    v=VAE(enc,dec)
    v.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),jit_compile=False)
    return v,enc,dec

vae,venc,vdec=build_vae(2)
venc.summary()
vdec.summary()
t0=time.time()
h=vae.fit(x_tr,x_tr,validation_data=(x_val,x_val),epochs=epochs_vae,batch_size=batch,verbose=2)
hist_vae=h.history
time_vae=time.time()-t0
params_vae=venc.count_params()+vdec.count_params()
print("VAE training time (s)",round(time_vae,2))
print("VAE history keys",list(hist_vae.keys()))

In [ ]:
ep=np.arange(1,len(hist_vae["loss"])+1)
fig,ax=plt.subplots(1,3,figsize=(15,4.2))
for a,(k,t) in zip(ax,[("rec","Reconstruction loss"),("kl","KL divergence"),("loss","Total loss")]):
    a.plot(ep,hist_vae[k],label="Training")
    a.plot(ep,hist_vae["val_"+k],label="Validation")
    a.set_xlabel("Epoch")
    a.set_ylabel(t)
    a.set_title(t)
    a.legend()
    a.grid(True)
save_fig("plot09_vae_train_val_loss")
loss_summary(hist_vae,"rec","val_rec","VAE reconstruction loss summary")
loss_summary(hist_vae,"kl","val_kl","VAE KL loss summary")
loss_summary(hist_vae,"loss","val_loss","VAE total loss summary")

zm_t,zl_t,z_t=[np.array(a) for a in venc.predict(x_test,batch_size=256,verbose=0)]
xh_vae=vdec.predict(zm_t,batch_size=256,verbose=0)
res_vae=score(x_test,xh_vae)
ev=vae.evaluate(x_test,x_test,batch_size=256,verbose=0,return_dict=True)
vae_df=pd.DataFrame([
{"VAE Metric":"Reconstruction Loss (BCE summed per image)","Value":round(float(ev["rec"]),4)},
{"VAE Metric":"KL Loss","Value":round(float(ev["kl"]),4)},
{"VAE Metric":"Total Loss","Value":round(float(ev["loss"]),4)},
{"VAE Metric":"Test MSE","Value":round(res_vae["MSE"],6)},
{"VAE Metric":"Test MAE","Value":round(res_vae["MAE"],6)},
{"VAE Metric":"Mean SSIM","Value":round(res_vae["SSIM"],6)}])
show(vae_df,"VAE test metrics (image metrics use the decoder output at the posterior mean)")
print("VAE parameters",params_vae)

plot_rows([x_test,xh_vae,np.abs(x_test-xh_vae)],["Original","VAE Reconstruction","Absolute Error"],"plot01c_original_vs_reconstructed_vae",n=10,cmaps=["gray","gray","hot"])

plot_latent_scatter(zm_t,y_test,"plot06_vae_latent_space","VAE latent space (posterior mean) coloured by digit label")
latent_stats(zm_t,y_test,"VAE latent space statistics (posterior mean)")
print("Mean posterior std per dimension",np.round(np.exp(0.5*zl_t).mean(axis=0),4))
print("Mean KL per latent dimension",np.round((-0.5*(1+zl_t-zm_t**2-np.exp(zl_t))).mean(axis=0),4))
print("Aggregate posterior mean",np.round(zm_t.mean(axis=0),4),"std",np.round(zm_t.std(axis=0),4),"(prior is mean 0, std 1)")

In [ ]:
rg=np.random.default_rng(7)
zg=rg.normal(0,1,size=(25,2)).astype("float32")
gen=vdec.predict(zg,batch_size=25,verbose=0)
plot_grid(gen,5,5,"plot07_vae_generated_samples","Images generated by sampling z from N(0,I)")
pw=lambda a:float(np.mean([np.mean((a[i]-a[j])**2) for i in range(len(a)) for j in range(i+1,len(a))]))
print("Mean pairwise MSE among 25 generated images (diversity)",round(pw(gen),5))
print("Mean pairwise MSE among 25 real test images",round(pw(x_test[:25]),5))
print("Mean pixel intensity generated",round(float(gen.mean()),4),"real",round(float(x_test.mean()),4))
print("Fraction of mid-grey pixels (0.2 to 0.8) generated",round(float(np.mean((gen>0.2)&(gen<0.8))),4),"real",round(float(np.mean((x_test>0.2)&(x_test<0.8))),4))
print("Per-image mean intensity of generated images",np.round(gen.mean(axis=(1,2,3)),3))
zg_big=rg.normal(0,1,size=(1000,2)).astype("float32")
gen_big=vdec.predict(zg_big,batch_size=250,verbose=0)
print("Std of pixel values across 1000 generated images (mean over pixels)",round(float(gen_big.std(axis=0).mean()),4),"real",round(float(x_test.std(axis=0).mean()),4))

alphas=np.round(np.arange(0,1.01,0.1),1)
pairs=[(3,8),(1,7),(0,6)]
rows_i=[]
labels_i=[]
for a,b in pairs:
    ia=int(np.where(y_test==a)[0][0])
    ib=int(np.where(y_test==b)[0][0])
    za=zm_t[ia]
    zb=zm_t[ib]
    zs=np.array([(1-al)*za+al*zb for al in alphas],dtype="float32")
    di=vdec.predict(zs,verbose=0)
    rows_i.append(di)
    labels_i.append("Interpolation from digit "+str(a)+" to digit "+str(b)+" (alpha 0 to 1)")
    steps=np.array([np.mean((di[k+1]-di[k])**2) for k in range(len(di)-1)])
    print("Pair",a,b,"z_A",np.round(za,3),"z_B",np.round(zb,3))
    print("  Consecutive-step MSE",np.round(steps,5))
    print("  Mean step",round(float(steps.mean()),5),"max step",round(float(steps.max()),5),"max/mean",round(float(steps.max()/steps.mean()),3))
    print("  MSE between first and last decoded image",round(float(np.mean((di[0]-di[-1])**2)),5))
plot_rows(rows_i,labels_i,"plot08_latent_space_interpolation",n=len(alphas))

In [ ]:
fc_models={16:(fc,fc_enc,res_fc,params_fc,time_fc)}
for dz in latent_dims:
    if dz==16:
        continue
    keras.utils.set_random_seed(seed)
    m,e=build_fc(dz)
    hh,tt=fit_model(m,flat(x_tr),flat(x_tr),flat(x_val),flat(x_val),epochs_study)
    xh=unflat(m.predict(flat(x_test),batch_size=256,verbose=0))
    fc_models[dz]=(m,e,score(x_test,xh),m.count_params(),tt)

lat_rows=[]
for dz in latent_dims:
    r=fc_models[dz][2]
    lat_rows.append({"Latent Dimension":dz,"MSE":round(r["MSE"],6),"MAE":round(r["MAE"],6),"SSIM":round(r["SSIM"],6),"Parameters":fc_models[dz][3],"Time (s)":round(fc_models[dz][4],2)})
lat_df=pd.DataFrame(lat_rows)
show(lat_df,"Latent dimension study (fully connected autoencoder)")

fig,ax=plt.subplots(1,2,figsize=(11,4.2))
ax[0].plot(lat_df["Latent Dimension"],lat_df["MSE"],marker="o")
ax[0].set_xscale("log",base=2)
ax[0].set_xticks(latent_dims)
ax[0].set_xticklabels(latent_dims)
ax[0].set_xlabel("Latent dimension")
ax[0].set_ylabel("Test MSE")
ax[0].set_title("Latent dimension versus reconstruction MSE")
ax[0].grid(True)
ax[1].plot(lat_df["Latent Dimension"],lat_df["SSIM"],marker="o",color="tab:red")
ax[1].set_xscale("log",base=2)
ax[1].set_xticks(latent_dims)
ax[1].set_xticklabels(latent_dims)
ax[1].set_xlabel("Latent dimension")
ax[1].set_ylabel("Mean SSIM")
ax[1].set_title("Latent dimension versus SSIM")
ax[1].grid(True)
save_fig("plot10_latent_dimension_vs_mse")

xh_by_dz=[unflat(fc_models[d][0].predict(flat(x_test[:10]),verbose=0)) for d in latent_dims]
plot_rows([x_test[:10]]+xh_by_dz,["Original"]+["FC-AE latent "+str(d) for d in latent_dims],"plot10b_latent_dimension_reconstructions",n=10)

z2=fc_models[2][1].predict(flat(x_test),batch_size=256,verbose=0)
plot_latent_scatter(z2,y_test,"plot06b_fcae_latent_space_dim2","Deterministic FC-AE latent space (dimension 2) coloured by digit label")
latent_stats(z2,y_test,"Deterministic FC-AE latent space statistics (dimension 2)")

In [ ]:
xh_dn_clean_target=xh_dn
err_sets={"FC AE":xh_fc,"Convolutional AE":xh_cae,"Denoising CAE (Gaussian sigma 0.2 input)":xh_dn,"VAE":xh_vae}
fig,ax=plt.subplots(2,2,figsize=(12,8))
err_store={}
for a,(name,xh) in zip(ax.ravel(),err_sets.items()):
    e=per_img_mse(x_test,xh)
    s=ssim_each(x_test,xh)
    err_store[name]=(e,s)
    a.hist(e,bins=50)
    a.axvline(e.mean(),linestyle="--",color="k",label="Mean")
    a.axvline(np.median(e),linestyle=":",color="r",label="Median")
    a.set_xlabel("Per-image reconstruction error (MSE)")
    a.set_ylabel("Number of test images")
    a.set_title(name)
    a.legend()
    a.grid(True)
save_fig("plot11_reconstruction_error_distribution")

stat_rows=[]
for name,(e,s) in err_store.items():
    q=np.percentile(e,[5,25,50,75,95,99])
    stat_rows.append({"Model":name,"Min":round(float(e.min()),6),"P5":round(float(q[0]),6),"P25":round(float(q[1]),6),"Median":round(float(q[2]),6),"Mean":round(float(e.mean()),6),"P75":round(float(q[3]),6),"P95":round(float(q[4]),6),"P99":round(float(q[5]),6),"Max":round(float(e.max()),6),"Std":round(float(e.std()),6),"Corr(error,SSIM)":round(float(np.corrcoef(e,s)[0,1]),4)})
show(pd.DataFrame(stat_rows),"Per-image reconstruction error statistics on the test set")

for name,xh in err_sets.items():
    e=err_store[name][0]
    s=err_store[name][1]
    idx=np.argsort(e)[::-1][:5]
    rows=[]
    for k,i in enumerate(idx):
        rows.append({"Sample":k+1,"Test Index":int(i),"Digit":int(y_test[i]),"Per-image MSE":round(float(e[i]),6),"SSIM":round(float(s[i]),4)})
    show(pd.DataFrame(rows),"Five highest-error test images: "+name)
    tag=name.split(" (")[0].lower().replace(" ","_")
    plot_rows([x_test[idx],xh[idx]],["Original","Reconstruction"],"plot12_high_error_"+tag,n=5)

In [ ]:
vae_time=time_vae
final=pd.DataFrame([
{"Model":"FC Autoencoder","MSE":round(res_fc["MSE"],6),"MAE":round(res_fc["MAE"],6),"SSIM":round(res_fc["SSIM"],6),"Parameters":params_fc,"Time (s)":round(time_fc,2)},
{"Model":"Conv. Autoencoder","MSE":round(res_cae["MSE"],6),"MAE":round(res_cae["MAE"],6),"SSIM":round(res_cae["SSIM"],6),"Parameters":params_cae,"Time (s)":round(time_cae,2)},
{"Model":"Denoising CAE (Gaussian sigma 0.2)","MSE":round(res_dn["MSE"],6),"MAE":round(res_dn["MAE"],6),"SSIM":round(res_dn["SSIM"],6),"Parameters":params_dn,"Time (s)":round(time_dg,2)},
{"Model":"VAE (latent 2)","MSE":round(res_vae["MSE"],6),"MAE":round(res_vae["MAE"],6),"SSIM":round(res_vae["SSIM"],6),"Parameters":params_vae,"Time (s)":round(vae_time,2)}])
show(final,"Consolidated results")

cmp3=final.iloc[:3][["Model","MSE","MAE","SSIM","Parameters"]]
show(cmp3,"Reconstruction comparison across models")
show(vae_df,"VAE metrics")
show(df_g[["Noise Level","MSE","MAE","SSIM"]],"Noise level versus reconstruction quality (Gaussian)")
show(df_s[["Noise Level","MSE","MAE","SSIM"]],"Noise level versus reconstruction quality (Salt-and-pepper)")
show(lat_df[["Latent Dimension","MSE","SSIM","Parameters"]],"Latent dimension study")
show(cmp_df,"Fully connected versus convolutional autoencoder")

shutil.make_archive("/kaggle/working/lab_outputs","zip",out)
for f in sorted(os.listdir(out)):
    print(f)
print("Saved zip archive at /kaggle/working/lab_outputs.zip")